In [ ]:
# Exploratory Data Analysis (EDA)

This notebook provides a practical EDA for the Kaggle *Harmonizing the Data of Your Data* repository.

It covers:
- Dataset inventory and file counts
- Basic schema checks for tabular files
- Missing-value and duplicate analysis
- Target/prediction column exploration (when available)
- Publication text corpus statistics (document and token lengths)
- Quick vocabulary snapshot from training publication text

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {ROOT}')
print(f'Data dir exists: {DATA_DIR.exists()} -> {DATA_DIR}')

In [ ]:
def safe_read_csv(path: Path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f'Could not read {path.name}: {exc}')
        return None


def load_pubtext_records(folder: Path):
    records = []
    if not folder.exists():
        return records

    for json_path in sorted(folder.glob('*.json')):
        if json_path.name == 'PubText.json':
            continue
        try:
            payload = json.loads(json_path.read_text(encoding='utf-8'))
        except Exception:
            continue

        pxd = json_path.stem.replace('_PubText', '')

        # Handle multiple likely shapes robustly
        if isinstance(payload, dict):
            text = payload.get('text') or payload.get('PubText') or payload.get('pub_text')
            if text is None:
                text = ' '.join(str(v) for v in payload.values() if isinstance(v, (str, int, float)))
        elif isinstance(payload, list):
            text = ' '.join(str(item) for item in payload)
        else:
            text = str(payload)

        records.append({'pxd': pxd, 'text': text})

    return records


def text_metrics(series: pd.Series):
    text_series = series.fillna('').astype(str)
    char_len = text_series.str.len()
    token_len = text_series.str.split().str.len()
    return pd.DataFrame(
        {
            'char_len': char_len,
            'token_len': token_len,
        }
    )

In [ ]:
sample_submission = safe_read_csv(DATA_DIR / 'SampleSubmission.csv')
metrics_csv = safe_read_csv(DATA_DIR / 'detailed_evaluation_metrics.csv')

train_pubtext_records = load_pubtext_records(DATA_DIR / 'TrainingPubText')
test_pubtext_records = load_pubtext_records(DATA_DIR / 'TestPubText')

summary = pd.DataFrame(
    {
        'artifact': [
            'SampleSubmission.csv',
            'detailed_evaluation_metrics.csv',
            'TrainingPubText JSON records',
            'TestPubText JSON records',
            'TrainingSDRF files',
        ],
        'available': [
            sample_submission is not None,
            metrics_csv is not None,
            len(train_pubtext_records) > 0,
            len(test_pubtext_records) > 0,
            (DATA_DIR / 'TrainingSDRFs').exists(),
        ],
        'count_or_rows': [
            None if sample_submission is None else len(sample_submission),
            None if metrics_csv is None else len(metrics_csv),
            len(train_pubtext_records),
            len(test_pubtext_records),
            len(list((DATA_DIR / 'TrainingSDRFs').glob('*'))) if (DATA_DIR / 'TrainingSDRFs').exists() else 0,
        ],
    }
)

summary

In [ ]:
if sample_submission is not None:
    print('SampleSubmission shape:', sample_submission.shape)
    display(sample_submission.head())

    missing_pct = (sample_submission.isna().mean() * 100).sort_values(ascending=False)
    display(pd.DataFrame({'missing_pct': missing_pct}).head(15))

    duplicate_rows = sample_submission.duplicated().sum()
    print('Duplicate rows in SampleSubmission:', int(duplicate_rows))

if metrics_csv is not None:
    print('\ndetailed_evaluation_metrics shape:', metrics_csv.shape)
    display(metrics_csv.head())

    num_cols = metrics_csv.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = metrics_csv.select_dtypes(exclude=[np.number]).columns.tolist()

    print(f'Numeric cols: {len(num_cols)} | Categorical/other cols: {len(cat_cols)}')
    if num_cols:
        display(metrics_csv[num_cols].describe().T.sort_values('std', ascending=False).head(20))

    if cat_cols:
        nunique = metrics_csv[cat_cols].nunique(dropna=False).sort_values(ascending=False)
        display(pd.DataFrame({'nunique': nunique}).head(20))

In [ ]:
if metrics_csv is not None:
    numeric_cols = metrics_csv.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        n_cols = min(3, len(numeric_cols))
        n_rows = int(np.ceil(len(numeric_cols[:9]) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
        axes = np.array(axes).reshape(-1)

        for idx, col in enumerate(numeric_cols[:9]):
            metrics_csv[col].dropna().plot(kind='hist', bins=30, ax=axes[idx], title=col)
            axes[idx].set_xlabel(col)

        for j in range(len(numeric_cols[:9]), len(axes)):
            axes[j].axis('off')

        plt.suptitle('Numeric Distribution Snapshot (first up to 9 columns)', y=1.02)
        plt.tight_layout()
        plt.show()

if sample_submission is not None:
    sub_num_cols = sample_submission.select_dtypes(include=[np.number]).columns.tolist()
    if sub_num_cols:
        corr = sample_submission[sub_num_cols].corr(numeric_only=True)
        if corr.shape[0] > 1:
            plt.figure(figsize=(8, 6))
            im = plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
            plt.title('Correlation Heatmap - SampleSubmission numeric columns')
            plt.colorbar(im, fraction=0.046, pad=0.04)
            plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
            plt.yticks(range(len(corr.columns)), corr.columns)
            plt.tight_layout()
            plt.show()

In [ ]:
train_pub_df = pd.DataFrame(train_pubtext_records)
test_pub_df = pd.DataFrame(test_pubtext_records)

print('Training publication documents:', len(train_pub_df))
print('Test publication documents:', len(test_pub_df))

if not train_pub_df.empty:
    train_text_stats = text_metrics(train_pub_df['text'])
    print('\nTraining text length summary:')
    display(train_text_stats.describe().T)

    plt.figure(figsize=(10, 4))
    train_text_stats['token_len'].clip(upper=train_text_stats['token_len'].quantile(0.99)).plot(
        kind='hist', bins=40, title='Training PubText token length distribution (99th pct clipped)'
    )
    plt.xlabel('Token count')
    plt.tight_layout()
    plt.show()

    token_pattern = re.compile(r"[A-Za-z][A-Za-z\-']+")
    stop_words = {
        'the', 'and', 'for', 'with', 'from', 'that', 'this', 'were', 'are', 'was', 'have', 'has',
        'into', 'using', 'used', 'their', 'our', 'these', 'those', 'than', 'such', 'also', 'can', 'may'
    }

    counter = Counter()
    for text in train_pub_df['text'].fillna(''):
        tokens = [t.lower() for t in token_pattern.findall(str(text))]
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
        counter.update(tokens)

    top_words = pd.DataFrame(counter.most_common(30), columns=['token', 'count'])
    display(top_words)

if not test_pub_df.empty:
    test_text_stats = text_metrics(test_pub_df['text'])
    print('\nTest text length summary:')
    display(test_text_stats.describe().T)